In [1]:
import torch
from transformers import AutoTokenizer, AutoModel


MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

e:\z_data\protein-representation-ood\.conda\envs\protein-ood\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu")

In [3]:
print("Device:", device)

Device: cuda


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [5]:
model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(device)

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
model.eval()

EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 320, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-5): 6 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=320, out_features=320, bias=True)
            (key): Linear(in_features=320, out_features=320, bias=True)
            (value): Linear(in_features=320, out_features=320, bias=True)
            (rotary_embeddings): RotaryEmbedding()
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=320, out_features=320, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=320, out_features=1280, bias=True)
        )
        (output): EsmOutput(
        

In [7]:
sequence = "MKTIIALSYIFCLVFADYKDDDDK"

In [8]:
inputs = tokenizer(
    sequence,
    return_tensors="pt",
    add_special_tokens=True,
)

print("\nInput IDs shape:")
print(inputs["input_ids"].shape)

print("\nTokens:")
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))


Input IDs shape:
torch.Size([1, 26])

Tokens:
['<cls>', 'M', 'K', 'T', 'I', 'I', 'A', 'L', 'S', 'Y', 'I', 'F', 'C', 'L', 'V', 'F', 'A', 'D', 'Y', 'K', 'D', 'D', 'D', 'D', 'K', '<eos>']


In [9]:
inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

In [10]:
inputs

{'input_ids': tensor([[ 0, 20, 15, 11, 12, 12,  5,  4,  8, 19, 12, 18, 23,  4,  7, 18,  5, 13,
          19, 15, 13, 13, 13, 13, 15,  2]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1]], device='cuda:0')}

In [11]:
with torch.no_grad():
    outputs = model(**inputs)

In [12]:
outputs[1].shape

torch.Size([1, 320])

In [13]:
residue_embeddings = outputs.last_hidden_state

In [14]:
residue_embeddings.shape

torch.Size([1, 26, 320])

In [15]:
protein_residue_embeddings = residue_embeddings[:, 1:-1, :]

In [16]:
protein_residue_embeddings.shape

torch.Size([1, 24, 320])

In [17]:
protein_embedding = protein_residue_embeddings.mean(dim=1)

In [18]:
protein_embedding.shape

torch.Size([1, 320])

-------------------------------------------------------------------------

In [19]:
import random
import torch
import torch.nn.functional as F

In [20]:
def get_esm_embedding(sequence, tokenizer, model, device):
    inputs = tokenizer(
        sequence,
        return_tensors="pt",
        add_special_tokens=True,
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    # remove <cls> and <eos>
    residue_embeddings = outputs.last_hidden_state[:, 1:-1, :]

    # protein-level mean pooling
    protein_embedding = residue_embeddings.mean(dim=1)

    return protein_embedding

In [21]:
original = "MKTIIALSYIFCLVFADYKDDDDK"

In [22]:
mutant = "MKTIIALSYIFCLVFADYKDEDDK"

In [23]:
random.seed(42)

shuffled_list = list(original)
random.shuffle(shuffled_list)
shuffled = "".join(shuffled_list)

print("Original:", original)
print("Mutant:  ", mutant)
print("Shuffled:", shuffled)

Original: MKTIIALSYIFCLVFADYKDDDDK
Mutant:   MKTIIALSYIFCLVFADYKDEDDK
Shuffled: AAKKVFYCFDKLDLITDIDSYMID


In [24]:
z_original = get_esm_embedding(
    original, tokenizer, model, device
)

z_mutant = get_esm_embedding(
    mutant, tokenizer, model, device
)

z_shuffled = get_esm_embedding(
    shuffled, tokenizer, model, device
)

In [27]:
sim_original_mutant = F.cosine_similarity(
    z_original,
    z_mutant
)

sim_original_shuffled = F.cosine_similarity(
    z_original,
    z_shuffled
).item()

sim_mutant_shuffled = F.cosine_similarity(
    z_mutant,
    z_shuffled
).item()

print("Original vs mutant:  ", sim_original_mutant)
print("Original vs shuffled:", sim_original_shuffled)
print("Mutant vs shuffled:  ", sim_mutant_shuffled)

Original vs mutant:   tensor([0.9965], device='cuda:0')
Original vs shuffled: 0.8821871876716614
Mutant vs shuffled:   0.8914119005203247


-------------------------------------------------

In [31]:
import numpy as np

data = np.load(
    "data/processed/esm2_t6_embeddings.npz",
    allow_pickle=True,
)

X = data["embeddings"]
acc = data["acc"]

In [32]:
print(X.shape)
print(X.dtype)
print("NaN:", np.isnan(X).sum())
print("Inf:", np.isinf(X).sum())
print("ACC:", acc.shape)

(28303, 320)
float32
NaN: 0
Inf: 0
ACC: (28303,)


In [33]:
data['embeddings'].shape

(28303, 320)

In [34]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd

LABEL_COLS = [
    "Membrane",
    "Cytoplasm",
    "Nucleus",
    "Extracellular",
    "Cell membrane",
    "Mitochondrion",
    "Plastid",
    "Endoplasmic reticulum",
    "Lysosome/Vacuole",
    "Golgi apparatus",
    "Peroxisome",
]

df = pd.read_csv('data/processed/deeploc_multilabel.csv')
y = df[LABEL_COLS].values.astype("float32")

train_mask = df["split_homology"] == "train"
val_mask   = df["split_homology"] == "val"
test_mask  = df["split_homology"] == "test"

X_train = torch.tensor(X[train_mask], dtype=torch.float32)
y_train = torch.tensor(y[train_mask], dtype=torch.float32)

X_val = torch.tensor(X[val_mask], dtype=torch.float32)
y_val = torch.tensor(y[val_mask], dtype=torch.float32)

X_test = torch.tensor(X[test_mask], dtype=torch.float32)
y_test = torch.tensor(y[test_mask], dtype=torch.float32)

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

torch.Size([17145, 320]) torch.Size([17145, 11])
torch.Size([5696, 320]) torch.Size([5696, 11])
torch.Size([5462, 320]) torch.Size([5462, 11])


In [35]:
train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=256,
    shuffle=True,
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=512,
    shuffle=False,
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=512,
    shuffle=False,
)

In [36]:
import torch.nn as nn

class ProteinMLP(nn.Module):
    def __init__(self, input_dim=320, hidden_dim=256, output_dim=11):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)

In [37]:
criterion = nn.BCEWithLogitsLoss()

In [38]:
model = ProteinMLP()

In [39]:
print(model)

ProteinMLP(
  (net): Sequential(
    (0): Linear(in_features=320, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=11, bias=True)
  )
)


In [40]:
import random
import numpy as np
import torch
import torch.nn as nn

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = ProteinMLP().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

print(device)

cuda


In [41]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
):
    model.train()

    total_loss = 0.0

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # 1. Clear old gradients
        optimizer.zero_grad()

        # 2. Forward
        logits = model(X_batch)

        # 3. Compute loss
        loss = criterion(logits, y_batch)

        # 4. Backward
        loss.backward()

        # 5. Update parameters
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    mean_loss = total_loss / len(loader.dataset)

    return mean_loss

In [42]:
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
)

In [43]:
@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
):
    model.eval()

    total_loss = 0.0

    all_probs = []
    all_targets = []

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)

        loss = criterion(
            logits,
            y_batch,
        )

        probs = torch.sigmoid(logits)

        total_loss += (
            loss.item() * X_batch.size(0)
        )

        all_probs.append(
            probs.cpu().numpy()
        )

        all_targets.append(
            y_batch.cpu().numpy()
        )

    probs = np.concatenate(
        all_probs,
        axis=0,
    )

    targets = np.concatenate(
        all_targets,
        axis=0,
    )

    mean_loss = (
        total_loss / len(loader.dataset)
    )

    macro_auc = roc_auc_score(
        targets,
        probs,
        average="macro",
    )

    micro_auc = roc_auc_score(
        targets,
        probs,
        average="micro",
    )

    predictions = (
        probs >= 0.5
    ).astype(int)

    macro_f1 = f1_score(
        targets,
        predictions,
        average="macro",
        zero_division=0,
    )

    micro_f1 = f1_score(
        targets,
        predictions,
        average="micro",
        zero_division=0,
    )

    return {
        "loss": mean_loss,
        "macro_auc": macro_auc,
        "micro_auc": micro_auc,
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
    }

In [44]:
NUM_EPOCHS = 30
PATIENCE = 5

best_val_auc = -np.inf
best_state = None

patience_counter = 0

history = []

In [45]:
for epoch in range(1, NUM_EPOCHS + 1):

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device,
    )

    val_metrics = evaluate(
        model,
        val_loader,
        criterion,
        device,
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        **val_metrics,
    })

    print(
        f"Epoch {epoch:02d} | "
        f"train loss={train_loss:.4f} | "
        f"val loss={val_metrics['loss']:.4f} | "
        f"macro AUC={val_metrics['macro_auc']:.4f} | "
        f"micro AUC={val_metrics['micro_auc']:.4f} | "
        f"macro F1={val_metrics['macro_f1']:.4f}"
    )

    # Early stopping based on validation macro AUROC
    if val_metrics["macro_auc"] > best_val_auc:

        best_val_auc = val_metrics["macro_auc"]

        best_state = {
            key: value.cpu().clone()
            for key, value
            in model.state_dict().items()
        }

        patience_counter = 0

    else:
        patience_counter += 1

        if patience_counter >= PATIENCE:
            print(
                f"\nEarly stopping at epoch {epoch}"
            )
            break

Epoch 01 | train loss=0.3910 | val loss=0.3000 | macro AUC=0.7400 | micro AUC=0.8456 | macro F1=0.1414
Epoch 02 | train loss=0.2839 | val loss=0.2643 | macro AUC=0.8059 | micro AUC=0.8854 | macro F1=0.2703
Epoch 03 | train loss=0.2613 | val loss=0.2500 | macro AUC=0.8328 | micro AUC=0.9002 | macro F1=0.2908
Epoch 04 | train loss=0.2496 | val loss=0.2423 | macro AUC=0.8457 | micro AUC=0.9070 | macro F1=0.3250
Epoch 05 | train loss=0.2428 | val loss=0.2363 | macro AUC=0.8537 | micro AUC=0.9119 | macro F1=0.3429
Epoch 06 | train loss=0.2380 | val loss=0.2327 | macro AUC=0.8588 | micro AUC=0.9152 | macro F1=0.3615
Epoch 07 | train loss=0.2338 | val loss=0.2298 | macro AUC=0.8626 | micro AUC=0.9171 | macro F1=0.3846
Epoch 08 | train loss=0.2308 | val loss=0.2275 | macro AUC=0.8656 | micro AUC=0.9192 | macro F1=0.3970
Epoch 09 | train loss=0.2281 | val loss=0.2272 | macro AUC=0.8682 | micro AUC=0.9203 | macro F1=0.4149
Epoch 10 | train loss=0.2255 | val loss=0.2241 | macro AUC=0.8702 | micro

In [46]:
model.load_state_dict(best_state)
model = model.to(device)

test_metrics = evaluate(
    model,
    test_loader,
    criterion,
    device,
)

print("\nBest validation macro AUC:")
print(best_val_auc)

print("\nTest metrics:")
for key, value in test_metrics.items():
    print(f"{key}: {value:.4f}")


Best validation macro AUC:
0.879189209880941

Test metrics:
loss: 0.2045
macro_auc: 0.8946
micro_auc: 0.9340
macro_f1: 0.4847
micro_f1: 0.6713


In [47]:
from sklearn.metrics import roc_auc_score, f1_score

def per_label_metrics(model, loader, device, label_names):
    model.eval()

    all_probs = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)

            logits = model(X_batch)
            probs = torch.sigmoid(logits)

            all_probs.append(probs.cpu().numpy())
            all_targets.append(y_batch.numpy())

    probs = np.concatenate(all_probs, axis=0)
    targets = np.concatenate(all_targets, axis=0)

    rows = []

    for i, label in enumerate(label_names):
        auc = roc_auc_score(
            targets[:, i],
            probs[:, i],
        )

        pred = (
            probs[:, i] >= 0.5
        ).astype(int)

        f1 = f1_score(
            targets[:, i],
            pred,
            zero_division=0,
        )

        rows.append({
            "label": label,
            "AUROC": auc,
            "F1@0.5": f1,
            "positives": int(targets[:, i].sum()),
        })

    return pd.DataFrame(rows)

In [48]:
per_label_df = per_label_metrics(
    model,
    test_loader,
    device,
    LABEL_COLS,
)

print(
    per_label_df.sort_values(
        "AUROC",
        ascending=False,
    )
)

                    label     AUROC    F1@0.5  positives
3           Extracellular  0.976138  0.810858        599
6                 Plastid  0.967582  0.593023        222
10             Peroxisome  0.923736  0.037037         53
5           Mitochondrion  0.917721  0.614499        516
0                Membrane  0.905184  0.768502       1496
4           Cell membrane  0.904600  0.638842        697
2                 Nucleus  0.898377  0.727663       2017
7   Endoplasmic reticulum  0.889778  0.387900        396
1               Cytoplasm  0.834958  0.659901       1831
8        Lysosome/Vacuole  0.816123  0.000000        243
9         Golgi apparatus  0.806067  0.093458        192


In [49]:
from sklearn.metrics import f1_score
import numpy as np
import torch


@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()

    all_probs = []
    all_targets = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)

        logits = model(X_batch)
        probs = torch.sigmoid(logits)

        all_probs.append(probs.cpu().numpy())
        all_targets.append(y_batch.numpy())

    probs = np.concatenate(all_probs, axis=0)
    targets = np.concatenate(all_targets, axis=0)

    return probs, targets

In [50]:
val_probs, val_targets = get_predictions(
    model,
    val_loader,
    device,
)

In [51]:
def find_best_thresholds(probs, targets):
    thresholds = []

    threshold_grid = np.arange(
        0.05,
        0.96,
        0.01,
    )

    for i in range(targets.shape[1]):
        best_threshold = 0.5
        best_f1 = -1

        for threshold in threshold_grid:
            pred = (
                probs[:, i] >= threshold
            ).astype(int)

            f1 = f1_score(
                targets[:, i],
                pred,
                zero_division=0,
            )

            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold

        thresholds.append(best_threshold)

    return np.array(thresholds)

In [52]:
best_thresholds = find_best_thresholds(
    val_probs,
    val_targets,
)

for label, threshold in zip(
    LABEL_COLS,
    best_thresholds,
):
    print(
        f"{label:25s} "
        f"threshold={threshold:.2f}"
    )

Membrane                  threshold=0.40
Cytoplasm                 threshold=0.38
Nucleus                   threshold=0.39
Extracellular             threshold=0.43
Cell membrane             threshold=0.36
Mitochondrion             threshold=0.32
Plastid                   threshold=0.21
Endoplasmic reticulum     threshold=0.17
Lysosome/Vacuole          threshold=0.09
Golgi apparatus           threshold=0.10
Peroxisome                threshold=0.06


In [53]:
test_probs, test_targets = get_predictions(
    model,
    test_loader,
    device,
)

test_pred_tuned = (
    test_probs >= best_thresholds[None, :]
).astype(int)

macro_f1_tuned = f1_score(
    test_targets,
    test_pred_tuned,
    average="macro",
    zero_division=0,
)

micro_f1_tuned = f1_score(
    test_targets,
    test_pred_tuned,
    average="micro",
    zero_division=0,
)

print("Tuned macro F1:", macro_f1_tuned)
print("Tuned micro F1:", micro_f1_tuned)

Tuned macro F1: 0.5589656502777152
Tuned micro F1: 0.6552764902446911


In [54]:
for i, label in enumerate(LABEL_COLS):
    f1 = f1_score(
        test_targets[:, i],
        test_pred_tuned[:, i],
        zero_division=0,
    )

    print(
        f"{label:25s} "
        f"F1={f1:.4f} "
        f"threshold={best_thresholds[i]:.2f}"
    )

Membrane                  F1=0.7658 threshold=0.40
Cytoplasm                 F1=0.6837 threshold=0.38
Nucleus                   F1=0.7574 threshold=0.39
Extracellular             F1=0.8212 threshold=0.43
Cell membrane             F1=0.6552 threshold=0.36
Mitochondrion             F1=0.6316 threshold=0.32
Plastid                   F1=0.6478 threshold=0.21
Endoplasmic reticulum     F1=0.5121 threshold=0.17
Lysosome/Vacuole          F1=0.2334 threshold=0.09
Golgi apparatus           F1=0.2138 threshold=0.10
Peroxisome                F1=0.2267 threshold=0.06


In [55]:
from sklearn.metrics import average_precision_score

In [56]:
macro_auprc = average_precision_score(
    test_targets,
    test_probs,
    average="macro",
)

micro_auprc = average_precision_score(
    test_targets,
    test_probs,
    average="micro",
)

print("Macro AUPRC:", macro_auprc)
print("Micro AUPRC:", micro_auprc)

Macro AUPRC: 0.5720163437908231
Micro AUPRC: 0.7537733899927039


In [57]:
from pathlib import Path

fasta_path = Path(
    "data/processed/deeploc_sequences.fasta"
)

with open(fasta_path, "r") as f:
    for _ in range(10):
        print(f.readline().rstrip())

>Q28165
MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDYGNGLESEELEPEELLLEPEPEPEPEEEPPRPRAP
PGAPGPGPGSGAPGNQEEEEESGLVEGDPGDGAIEDPELEAIKARVREMEEEAEKLKELQNEVEKQMNMSPPPGNAGPVI
MSIEEKMEADARSIYVGNVDYGATAEELEAHFHGCGSVNRVTILCDKFSGHPKGFAYIEFSDKESVRTSLALDESLFRGR
QIKVIPKRTNRPGISTTDRGFPRARYRARTTNYNSSRSRFYSGFNSRPRGRVYRGRARATSWYSPY
>Q86U42
MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDYGNGLESEELEPEELLLEPEPEPEPEEEPPRPRAP
PGAPGPGPGSGAPGSQEEEEEPGLVEGDPGDGAIEDPELEAIKARVREMEEEAEKLKELQNEVEKQMNMSPPPGNAGPVI
MSIEEKMEADARSIYVGNVDYGATAEELEAHFHGCGSVNRVTILCDKFSGHPKGFAYIEFSDKESVRTSLALDESLFRGR
QIKVIPKRTNRPGISTTDRGFPRARYRARTTNYNSSRSRFYSGFNSRPRGRVYRGRARATSWYSPY


In [58]:
count = 0

with open(fasta_path, "r") as f:
    for line in f:
        if line.startswith(">"):
            count += 1

print(count)

28303


In [59]:
import pandas as pd

cols = [
    "query",
    "target",
    "fident",
    "alnlen",
    "qcov",
    "tcov",
    "evalue",
    "bits",
]

hits = pd.read_csv(
    "data/mmseqs/all_vs_all.tsv",
    sep="\t",
    names=cols,
)

print(hits.shape)
print(hits.head())

(623566, 8)
    query  target  fident  alnlen   qcov   tcov        evalue  bits
0  Q11193  Q11193   1.000     531  1.000  1.000  0.000000e+00  1125
1  Q14676  Q14676   1.000    2089  1.000  1.000  0.000000e+00  3962
2  Q14676  Q6ZW49   0.326     207  0.093  0.194  1.368000e-21   104
3  Q14676  Q6NZQ4   0.326     207  0.093  0.196  3.118000e-21   103
4  Q14676  O14079   0.234     152  0.072  0.244  2.002000e-04    47


In [60]:
hits = hits[hits["query"] != hits["target"]].copy()

print(hits.shape)

(595264, 8)


In [61]:
hits["min_cov"] = hits[["qcov", "tcov"]].min(axis=1)

In [62]:
print(
    hits[
        ["fident", "qcov", "tcov", "min_cov", "evalue", "bits"]
    ].describe(
        percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

              fident           qcov           tcov        min_cov  \
count  595264.000000  595264.000000  595264.000000  595264.000000   
mean        0.368193       0.517972       0.512649       0.447685   
std         0.140469       0.298366       0.299763       0.298206   
min         0.158000       0.002000       0.002000       0.002000   
10%         0.245000       0.134000       0.129000       0.099000   
25%         0.277000       0.252000       0.245000       0.189000   
50%         0.328000       0.488000       0.479000       0.372000   
75%         0.408000       0.796000       0.791000       0.713000   
90%         0.542000       0.946000       0.946000       0.912000   
95%         0.673000       0.988000       0.988000       0.976000   
99%         0.932000       1.000000       1.000000       1.000000   
max         1.000000       1.000000       1.000000       1.000000   

             evalue           bits  
count  5.952640e+05  595264.000000  
mean   2.333316e-05     165.

In [63]:
for identity, coverage in [
    (0.30, 0.30),
    (0.30, 0.50),
    (0.40, 0.50),
]:
    subset = hits[
        (hits["fident"] >= identity)
        & (hits["min_cov"] >= coverage)
    ]

    print(
        f"identity >= {identity}, "
        f"coverage >= {coverage}: "
        f"{len(subset)} directed edges"
    )

identity >= 0.3, coverage >= 0.3: 220334 directed edges
identity >= 0.3, coverage >= 0.5: 160071 directed edges
identity >= 0.4, coverage >= 0.5: 94098 directed edges


In [64]:
candidate = hits[
    (hits["fident"] >= 0.30)
    & (hits["min_cov"] >= 0.50)
]

degree = candidate.groupby("query").size()

print(degree.describe())
print("Proteins with neighbors:", degree.index.nunique())
print(
    "Proteins with no neighbors:",
    28303 - degree.index.nunique(),
)

count    19975.000000
mean         8.013567
std         15.017305
min          1.000000
25%          1.000000
50%          3.000000
75%          7.000000
max         99.000000
dtype: float64
Proteins with neighbors: 19975
Proteins with no neighbors: 8328


In [65]:
df = pd.read_csv('data/processed/deeploc_multilabel.csv')

In [66]:
candidate = hits[
    (hits["fident"] >= 0.30)
    & (hits["min_cov"] >= 0.50)
].copy()

split_map = dict(
    zip(
        df["ACC"],
        df["split_homology"]
    )
)

candidate["query_split"] = candidate["query"].map(split_map)
candidate["target_split"] = candidate["target"].map(split_map)

print(
    pd.crosstab(
        candidate["query_split"],
        candidate["target_split"]
    )
)

target_split   test  train    val
query_split                      
test          26252   3502   1076
train          3637  96463   3884
val            1019   3793  20445


In [67]:
val_to_train = candidate[
    (candidate["query_split"] == "val")
    & (candidate["target_split"] == "train")
]

test_to_train = candidate[
    (candidate["query_split"] == "test")
    & (candidate["target_split"] == "train")
]

n_val = (df["split_homology"] == "val").sum()
n_test = (df["split_homology"] == "test").sum()

val_connected = val_to_train["query"].nunique()
test_connected = test_to_train["query"].nunique()

print(
    f"Val proteins with ≥1 train neighbor: "
    f"{val_connected}/{n_val}"
)

print(
    f"Test proteins with ≥1 train neighbor: "
    f"{test_connected}/{n_test}"
)

Val proteins with ≥1 train neighbor: 948/5696
Test proteins with ≥1 train neighbor: 940/5462


In [68]:
train_edges = candidate[
    (candidate["query_split"] == "train")
    & (candidate["target_split"] == "train")
].copy()

print("Train-train edges:", len(train_edges))

Train-train edges: 96463


In [69]:
LABEL_COLS = [
    "Membrane",
    "Cytoplasm",
    "Nucleus",
    "Extracellular",
    "Cell membrane",
    "Mitochondrion",
    "Plastid",
    "Endoplasmic reticulum",
    "Lysosome/Vacuole",
    "Golgi apparatus",
    "Peroxisome",
]

label_map = {
    row["ACC"]: row[LABEL_COLS].to_numpy(dtype=float)
    for _, row in df.iterrows()
}

In [70]:
import numpy as np

def share_any_label(a, b):
    ya = label_map[a]
    yb = label_map[b]

    return np.any((ya == 1) & (yb == 1))

In [71]:
edge_shared_rate = np.mean([
    share_any_label(q, t)
    for q, t in zip(
        train_edges["query"],
        train_edges["target"]
    )
])

print("Sequence-edge shared-label rate:",
      edge_shared_rate)

Sequence-edge shared-label rate: 0.8711734032737941


In [72]:
rng = np.random.default_rng(42)

train_acc = df.loc[
    df["split_homology"] == "train",
    "ACC"
].to_numpy()

n_pairs = len(train_edges)

random_q = rng.choice(
    train_acc,
    size=n_pairs,
    replace=True,
)

random_t = rng.choice(
    train_acc,
    size=n_pairs,
    replace=True,
)

random_shared_rate = np.mean([
    share_any_label(q, t)
    for q, t in zip(random_q, random_t)
])

print("Random-pair shared-label rate:",
      random_shared_rate)

Random-pair shared-label rate: 0.3420482464779242


In [78]:
candidate["weight"] = (
    candidate["fident"]
    * candidate["min_cov"]
)

In [74]:
train_acc_set = set(
    df.loc[
        df["split_homology"] == "train",
        "ACC"
    ]
)

edges_to_train = candidate[
    candidate["target"].isin(train_acc_set)
].copy()

In [79]:
candidate

,query,target,fident,alnlen,qcov,tcov,evalue,bits,min_cov,query_split,target_split,weight
7,P49682-2,P49682,0.997,365,0.880,0.992,1.109000e-236,727,0.880,train,train,0.877360
8,P49682-2,P61073,0.388,285,0.687,0.807,2.275000e-59,213,0.687,train,train,0.266556
9,P49682-2,P51684,0.391,294,0.696,0.786,4.319000e-57,206,0.696,train,train,0.272136
10,P49682-2,O54689,0.372,302,0.716,0.823,8.135000e-55,199,0.716,train,train,0.266352
11,P49682-2,P41597,0.348,312,0.752,0.794,9.543000e-54,196,0.752,train,train,0.261696
...,...,...,...,...,...,...,...,...,...,...,...,...
623517,B9DFX7,O59666,0.361,650,0.736,0.700,6.752000e-92,325,0.700,test,test,0.252700
623518,B9DFX7,Q04656-5,0.308,845,0.905,0.594,1.779000e-87,312,0.594,test,train,0.182952
623563,Q41738,Q41739,0.913,354,1.000,1.000,2.534000e-199,616,1.000,train,train,0.913000
623564,Q41738,Q38814,0.832,299,0.842,0.857,4.496000e-160,502,0.842,train,train,0.700544


## Fixed-neighbor aggregation baseline

For every protein $i$, aggregate **training-set neighbors only**:

$$n_i = \frac{\sum_{j \in N_{train}(i)} w_{ij}z_j}{\sum_{j \in N_{train}(i)} w_{ij}}, \qquad w_{ij}=fident_{ij}\,mincov_{ij}$$

Then concatenate the original ESM2 embedding and the fixed neighbor vector, $z'_i=[z_i^{ESM}\Vert n_i]$, and train a $640\rightarrow256\rightarrow11$ MLP. The graph aggregation has no trainable parameters. Validation/test proteins may read ESM2 embeddings from training proteins, but never from validation/test neighbors. Proteins without a training neighbor receive an all-zero neighbor vector.

In [ ]:
from scipy.sparse import csr_matrix

# Align embeddings by accession instead of assuming that the NPZ and CSV
# happen to have the same row order.
embedding_acc = pd.Index(np.asarray(acc).astype(str), name="ACC")
dataset_acc = pd.Index(df["ACC"].astype(str), name="ACC")

if not embedding_acc.is_unique:
    raise ValueError("Duplicate accessions found in the embedding file.")
if not dataset_acc.is_unique:
    raise ValueError("Duplicate accessions found in the processed dataset.")

missing_embeddings = dataset_acc.difference(embedding_acc)
if len(missing_embeddings):
    raise ValueError(
        f"Missing embeddings for {len(missing_embeddings)} proteins; "
        f"examples: {missing_embeddings[:5].tolist()}"
    )

embedding_position = pd.Series(
    np.arange(len(embedding_acc)),
    index=embedding_acc,
)
aligned_positions = embedding_position.loc[dataset_acc].to_numpy()
X_esm = np.asarray(X[aligned_positions], dtype=np.float32)

# Keep one edge per ordered protein pair and only allow training targets.
train_accessions = set(dataset_acc[train_mask.to_numpy()])
train_neighbor_edges = candidate.loc[
    candidate["target"].astype(str).isin(train_accessions),
    ["query", "target", "fident", "min_cov"],
].copy()
train_neighbor_edges[["query", "target"]] = (
    train_neighbor_edges[["query", "target"]].astype(str)
)
train_neighbor_edges["weight"] = (
    train_neighbor_edges["fident"]
    * train_neighbor_edges["min_cov"]
).astype(np.float32)
train_neighbor_edges = (
    train_neighbor_edges
    .groupby(["query", "target"], as_index=False, sort=False)["weight"]
    .max()
)

protein_index = pd.Series(np.arange(len(dataset_acc)), index=dataset_acc)
query_idx = train_neighbor_edges["query"].map(protein_index)
target_idx = train_neighbor_edges["target"].map(protein_index)
if query_idx.isna().any() or target_idx.isna().any():
    raise ValueError("At least one MMseqs2 accession is absent from the dataset.")

weighted_adjacency = csr_matrix(
    (
        train_neighbor_edges["weight"].to_numpy(np.float32),
        (query_idx.to_numpy(int), target_idx.to_numpy(int)),
    ),
    shape=(len(df), len(df)),
    dtype=np.float32,
)

neighbor_weight_sum = np.asarray(weighted_adjacency.sum(axis=1)).ravel()
neighbor_count = np.diff(weighted_adjacency.indptr)
has_train_neighbor = neighbor_weight_sum > 0

neighbor_embeddings = np.zeros_like(X_esm, dtype=np.float32)
neighbor_embeddings[has_train_neighbor] = (
    (weighted_adjacency @ X_esm)[has_train_neighbor]
    / neighbor_weight_sum[has_train_neighbor, None]
)
X_fixed_neighbor = np.concatenate(
    [X_esm, neighbor_embeddings],
    axis=1,
).astype(np.float32, copy=False)

assert X_fixed_neighbor.shape == (len(df), 640)
assert np.isfinite(X_fixed_neighbor).all()
assert np.all(neighbor_embeddings[~has_train_neighbor] == 0)

aggregation_diagnostics = (
    pd.DataFrame({
        "split": df["split_homology"],
        "has_train_neighbor": has_train_neighbor,
        "n_train_neighbors": neighbor_count,
    })
    .groupby("split", sort=False)
    .agg(
        proteins=("has_train_neighbor", "size"),
        proteins_with_train_neighbor=("has_train_neighbor", "sum"),
        train_neighbor_coverage=("has_train_neighbor", "mean"),
        mean_train_neighbors=("n_train_neighbors", "mean"),
    )
)

print(f"Training-target edges used: {weighted_adjacency.nnz:,}")
print(f"Fixed-neighbor feature matrix: {X_fixed_neighbor.shape}")
display(aggregation_diagnostics)

In [ ]:
train_mask_np = train_mask.to_numpy()
val_mask_np = val_mask.to_numpy()
test_mask_np = test_mask.to_numpy()

fixed_train_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(X_fixed_neighbor[train_mask_np]),
        torch.from_numpy(y[train_mask_np]),
    ),
    batch_size=256,
    shuffle=True,
)
fixed_val_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(X_fixed_neighbor[val_mask_np]),
        torch.from_numpy(y[val_mask_np]),
    ),
    batch_size=512,
    shuffle=False,
)
fixed_test_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(X_fixed_neighbor[test_mask_np]),
        torch.from_numpy(y[test_mask_np]),
    ),
    batch_size=512,
    shuffle=False,
)

fixed_batch_x, fixed_batch_y = next(iter(fixed_train_loader))
print(fixed_batch_x.shape, fixed_batch_y.shape)

In [ ]:
# Preserve the ESM2-only result for the comparison table below when the
# notebook has been run top-to-bottom.
esm_mlp_reference = None
if "test_metrics" in globals():
    esm_mlp_reference = {"model": "ESM2 + MLP", **test_metrics}
    if "macro_auprc" in globals():
        esm_mlp_reference["macro_auprc"] = macro_auprc
        esm_mlp_reference["micro_auprc"] = micro_auprc
    if "macro_f1_tuned" in globals():
        esm_mlp_reference["macro_f1_tuned"] = macro_f1_tuned
        esm_mlp_reference["micro_f1_tuned"] = micro_f1_tuned

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

fixed_model = ProteinMLP(
    input_dim=640,
    hidden_dim=256,
    output_dim=len(LABEL_COLS),
).to(device)
fixed_criterion = nn.BCEWithLogitsLoss()
fixed_optimizer = torch.optim.AdamW(
    fixed_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

print(fixed_model)

In [ ]:
FIXED_NUM_EPOCHS = 30
FIXED_PATIENCE = 5

fixed_best_val_auc = -np.inf
fixed_best_state = None
fixed_patience_counter = 0
fixed_history = []

for epoch in range(1, FIXED_NUM_EPOCHS + 1):
    train_loss = train_one_epoch(
        fixed_model,
        fixed_train_loader,
        fixed_criterion,
        fixed_optimizer,
        device,
    )
    val_metrics = evaluate(
        fixed_model,
        fixed_val_loader,
        fixed_criterion,
        device,
    )
    fixed_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        **val_metrics,
    })

    print(
        f"Epoch {epoch:02d} | train loss={train_loss:.4f} | "
        f"val loss={val_metrics['loss']:.4f} | "
        f"macro AUC={val_metrics['macro_auc']:.4f} | "
        f"micro AUC={val_metrics['micro_auc']:.4f} | "
        f"macro F1={val_metrics['macro_f1']:.4f}"
    )

    if val_metrics["macro_auc"] > fixed_best_val_auc:
        fixed_best_val_auc = val_metrics["macro_auc"]
        fixed_best_state = {
            key: value.detach().cpu().clone()
            for key, value in fixed_model.state_dict().items()
        }
        fixed_patience_counter = 0
    else:
        fixed_patience_counter += 1
        if fixed_patience_counter >= FIXED_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

if fixed_best_state is None:
    raise RuntimeError("Training finished without a valid checkpoint.")

In [ ]:
from sklearn.metrics import average_precision_score

fixed_model.load_state_dict(fixed_best_state)
fixed_model = fixed_model.to(device)

fixed_test_metrics = evaluate(
    fixed_model,
    fixed_test_loader,
    fixed_criterion,
    device,
)
fixed_val_probs, fixed_val_targets = get_predictions(
    fixed_model, fixed_val_loader, device
)
fixed_test_probs, fixed_test_targets = get_predictions(
    fixed_model, fixed_test_loader, device
)
fixed_thresholds = find_best_thresholds(
    fixed_val_probs, fixed_val_targets
)
fixed_test_pred_tuned = (
    fixed_test_probs >= fixed_thresholds[None, :]
).astype(int)

fixed_macro_f1_tuned = f1_score(
    fixed_test_targets,
    fixed_test_pred_tuned,
    average="macro",
    zero_division=0,
)
fixed_micro_f1_tuned = f1_score(
    fixed_test_targets,
    fixed_test_pred_tuned,
    average="micro",
    zero_division=0,
)
fixed_macro_auprc = average_precision_score(
    fixed_test_targets, fixed_test_probs, average="macro"
)
fixed_micro_auprc = average_precision_score(
    fixed_test_targets, fixed_test_probs, average="micro"
)

fixed_summary = {
    "model": "ESM2 + fixed neighbor aggregation",
    **fixed_test_metrics,
    "macro_auprc": fixed_macro_auprc,
    "micro_auprc": fixed_micro_auprc,
    "macro_f1_tuned": fixed_macro_f1_tuned,
    "micro_f1_tuned": fixed_micro_f1_tuned,
}
comparison_rows = [fixed_summary]
if esm_mlp_reference is not None:
    comparison_rows.insert(0, esm_mlp_reference)

comparison_df = pd.DataFrame(comparison_rows).set_index("model")
display(comparison_df.round(4))

In [ ]:
fixed_per_label_df = per_label_metrics(
    fixed_model,
    fixed_test_loader,
    device,
    LABEL_COLS,
)
fixed_per_label_df["threshold"] = fixed_thresholds
fixed_per_label_df["F1@tuned"] = [
    f1_score(
        fixed_test_targets[:, label_idx],
        fixed_test_pred_tuned[:, label_idx],
        zero_division=0,
    )
    for label_idx in range(len(LABEL_COLS))
]
display(fixed_per_label_df.round(4))

## Does fixed aggregation help proteins that actually have a training homolog?

The overall test score is dominated by remote proteins without a training neighbor. We therefore compare the same two trained models on two disjoint test subsets:

1. **Has train neighbor**: the test protein has at least one candidate edge to a training protein.
2. **No train neighbor**: the test protein has no candidate edge to any training protein.

All model selection and per-label thresholds remain validation-only. No threshold is fitted on either test subset. For robustness, a label is included in AUROC/AUPRC/F1 only when that subset contains both positive and negative examples for the label.

In [ ]:
def multilabel_subset_metrics(targets, probs, thresholds):
    valid_label_idx = np.array([
        label_idx
        for label_idx in range(targets.shape[1])
        if np.unique(targets[:, label_idx]).size == 2
    ])
    if len(valid_label_idx) == 0:
        raise ValueError("This subset has no label with both classes present.")

    subset_targets = targets[:, valid_label_idx]
    subset_probs = probs[:, valid_label_idx]
    pred_at_05 = (subset_probs >= 0.5).astype(int)
    pred_tuned = (
        subset_probs >= thresholds[valid_label_idx][None, :]
    ).astype(int)

    return {
        "n": len(targets),
        "labels_evaluated": len(valid_label_idx),
        "macro_auroc": roc_auc_score(
            subset_targets, subset_probs, average="macro"
        ),
        "micro_auroc": roc_auc_score(
            subset_targets, subset_probs, average="micro"
        ),
        "macro_auprc": average_precision_score(
            subset_targets, subset_probs, average="macro"
        ),
        "micro_auprc": average_precision_score(
            subset_targets, subset_probs, average="micro"
        ),
        "macro_f1_at_05": f1_score(
            subset_targets, pred_at_05, average="macro", zero_division=0
        ),
        "micro_f1_at_05": f1_score(
            subset_targets, pred_at_05, average="micro", zero_division=0
        ),
        "macro_f1_tuned": f1_score(
            subset_targets, pred_tuned, average="macro", zero_division=0
        ),
        "micro_f1_tuned": f1_score(
            subset_targets, pred_tuned, average="micro", zero_division=0
        ),
    }

if not np.array_equal(test_targets, fixed_test_targets):
    raise ValueError("The two test prediction arrays use different sample orders.")

test_has_train_neighbor = has_train_neighbor[test_mask_np]
if len(test_has_train_neighbor) != len(test_targets):
    raise ValueError("Neighbor mask and test predictions have different lengths.")

test_subsets = {
    "Has train neighbor": test_has_train_neighbor,
    "No train neighbor": ~test_has_train_neighbor,
}
model_predictions = {
    "ESM2 + MLP": (test_probs, best_thresholds),
    "ESM2 + fixed neighbor": (fixed_test_probs, fixed_thresholds),
}

subgroup_rows = []
for subset_name, subset_mask in test_subsets.items():
    for model_name, (probs, thresholds) in model_predictions.items():
        subgroup_rows.append({
            "test_subset": subset_name,
            "model": model_name,
            **multilabel_subset_metrics(
                test_targets[subset_mask],
                probs[subset_mask],
                thresholds,
            ),
        })

subgroup_results = (
    pd.DataFrame(subgroup_rows)
    .set_index(["test_subset", "model"])
)
display(subgroup_results.round(4))

In [ ]:
metric_cols = [
    "macro_auroc",
    "micro_auroc",
    "macro_auprc",
    "micro_auprc",
    "macro_f1_at_05",
    "micro_f1_at_05",
    "macro_f1_tuned",
    "micro_f1_tuned",
]
delta_rows = []
for subset_name in test_subsets:
    esm_metrics = subgroup_results.loc[(subset_name, "ESM2 + MLP")]
    neighbor_metrics = subgroup_results.loc[
        (subset_name, "ESM2 + fixed neighbor")
    ]
    delta_rows.append({
        "test_subset": subset_name,
        "n": int(esm_metrics["n"]),
        **{
            f"delta_{metric}": neighbor_metrics[metric] - esm_metrics[metric]
            for metric in metric_cols
        },
    })

subgroup_deltas = pd.DataFrame(delta_rows).set_index("test_subset")
print("Delta = fixed neighbor - ESM2; positive values favor neighbor aggregation.")
display(subgroup_deltas.round(4))

macro_auroc_table = (
    subgroup_results["macro_auroc"]
    .unstack("model")
    .rename(columns={
        "ESM2 + MLP": "ESM2",
        "ESM2 + fixed neighbor": "+ Neighbor",
    })
)
macro_auroc_table["Delta"] = (
    macro_auroc_table["+ Neighbor"] - macro_auroc_table["ESM2"]
)
display(macro_auroc_table.round(4))

In [ ]:
connected_mask = test_has_train_neighbor
connected_targets = test_targets[connected_mask]
connected_rows = []

for label_idx, label in enumerate(LABEL_COLS):
    label_targets = connected_targets[:, label_idx]
    if np.unique(label_targets).size < 2:
        continue

    esm_label_probs = test_probs[connected_mask, label_idx]
    neighbor_label_probs = fixed_test_probs[connected_mask, label_idx]
    esm_label_pred = (
        esm_label_probs >= best_thresholds[label_idx]
    ).astype(int)
    neighbor_label_pred = (
        neighbor_label_probs >= fixed_thresholds[label_idx]
    ).astype(int)

    esm_auc = roc_auc_score(label_targets, esm_label_probs)
    neighbor_auc = roc_auc_score(label_targets, neighbor_label_probs)
    esm_ap = average_precision_score(label_targets, esm_label_probs)
    neighbor_ap = average_precision_score(label_targets, neighbor_label_probs)
    esm_f1 = f1_score(label_targets, esm_label_pred, zero_division=0)
    neighbor_f1 = f1_score(
        label_targets, neighbor_label_pred, zero_division=0
    )

    connected_rows.append({
        "label": label,
        "positives": int(label_targets.sum()),
        "ESM2_AUROC": esm_auc,
        "Neighbor_AUROC": neighbor_auc,
        "delta_AUROC": neighbor_auc - esm_auc,
        "ESM2_AUPRC": esm_ap,
        "Neighbor_AUPRC": neighbor_ap,
        "delta_AUPRC": neighbor_ap - esm_ap,
        "ESM2_F1_tuned": esm_f1,
        "Neighbor_F1_tuned": neighbor_f1,
        "delta_F1_tuned": neighbor_f1 - esm_f1,
    })

connected_per_label = (
    pd.DataFrame(connected_rows)
    .sort_values("delta_AUROC", ascending=False)
)
display(connected_per_label.round(4))

### Interpretation of the seed-42 run

Among the **940 test proteins with a training neighbor**, fixed aggregation changes Macro AUROC from **0.9005 to 0.9018** ($+0.0013$) and Macro AUPRC from **0.6182 to 0.6281** ($+0.0099$). Macro F1 at the fixed 0.5 cutoff increases by $0.0129$, but validation-tuned Macro F1 decreases by $0.0300$ and both micro ranking metrics decrease slightly. This is mixed rather than decisive evidence of benefit.

Among the **4,522 proteins without a training neighbor**, Macro AUROC decreases by $0.0014$ and Macro AUPRC decreases by $0.0117$. Their neighbor vector is zero, but their predictions can still change because the 640D model was trained under a different feature distribution and learned different weights.

The per-label results are heterogeneous: the largest connected-subset ranking gains occur for Golgi apparatus, Lysosome/Vacuole, and Peroxisome, whereas several common labels do not improve. Therefore the appropriate conclusion is that fixed neighbor averaging contains a weak signal for some connected proteins/classes, but it is not a reliable general improvement. Multiple seeds or paired bootstrap confidence intervals are needed before treating the small aggregate differences as significant.

## Frozen-base residual neighbor model

Let the already-trained ESM2 baseline produce $l_i^{base}=f(z_i)$. A separate network predicts only a correction, $\Delta l_i=g([z_i\Vert n_i])$, and the final logits are

$$l_i=l_i^{base}+m_i\Delta l_i, \qquad m_i=\mathbb{1}[N_{train}(i)\neq\varnothing].$$

To make the no-neighbor guarantee literal rather than approximate, the baseline branch is copied from the selected ESM2-only checkpoint and frozen. The correction output layer is initialized to zero. The correction network is trained only on training proteins with a training neighbor and selected using only the corresponding validation subset. Thus, when $m_i=0$, both logits and probabilities are numerically identical to the original baseline.

In [ ]:
import copy

class ResidualNeighborModel(nn.Module):
    def __init__(self, base_model, embedding_dim=320, hidden_dim=256, output_dim=11):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.base_model = copy.deepcopy(base_model)
        for parameter in self.base_model.parameters():
            parameter.requires_grad = False

        self.correction_net = nn.Sequential(
            nn.Linear(2 * embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim),
        )
        # The complete residual model starts as the exact ESM2 baseline.
        nn.init.zeros_(self.correction_net[-1].weight)
        nn.init.zeros_(self.correction_net[-1].bias)

    def train(self, mode=True):
        super().train(mode)
        # Frozen dropout must stay disabled in the baseline branch.
        self.base_model.eval()
        return self

    def forward(self, features, has_neighbor):
        z_i = features[:, :self.embedding_dim]
        with torch.no_grad():
            base_logits = self.base_model(z_i)
        delta_logits = self.correction_net(features)
        gate = has_neighbor.to(delta_logits.dtype).reshape(-1, 1)
        return base_logits + gate * delta_logits


def make_residual_loader(row_mask, batch_size, shuffle):
    row_indices = np.flatnonzero(np.asarray(row_mask, dtype=bool))
    return DataLoader(
        TensorDataset(
            torch.from_numpy(X_fixed_neighbor[row_indices]),
            torch.from_numpy(y[row_indices]),
            torch.from_numpy(has_train_neighbor[row_indices].astype(np.float32)),
        ),
        batch_size=batch_size,
        shuffle=shuffle,
    )


connected_train_mask = train_mask_np & has_train_neighbor
connected_val_mask = val_mask_np & has_train_neighbor
residual_train_loader = make_residual_loader(
    connected_train_mask, batch_size=256, shuffle=True
)
residual_val_connected_loader = make_residual_loader(
    connected_val_mask, batch_size=512, shuffle=False
)
residual_test_loader = make_residual_loader(
    test_mask_np, batch_size=512, shuffle=False
)

print("Connected training proteins:", connected_train_mask.sum())
print("Connected validation proteins:", connected_val_mask.sum())
print("Test proteins:", test_mask_np.sum())

In [ ]:
def train_residual_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for features, targets, has_neighbor_batch in loader:
        features = features.to(device)
        targets = targets.to(device)
        has_neighbor_batch = has_neighbor_batch.to(device)

        optimizer.zero_grad()
        logits = model(features, has_neighbor_batch)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(features)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def get_residual_predictions(model, loader, device):
    model.eval()
    all_probs, all_targets, all_masks = [], [], []

    for features, targets, has_neighbor_batch in loader:
        logits = model(
            features.to(device),
            has_neighbor_batch.to(device),
        )
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_targets.append(targets.numpy())
        all_masks.append(has_neighbor_batch.numpy().astype(bool))

    return (
        np.concatenate(all_probs),
        np.concatenate(all_targets),
        np.concatenate(all_masks),
    )


@torch.no_grad()
def evaluate_residual(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_probs, all_targets = [], []

    for features, targets, has_neighbor_batch in loader:
        features = features.to(device)
        targets_device = targets.to(device)
        logits = model(features, has_neighbor_batch.to(device))
        loss = criterion(logits, targets_device)
        total_loss += loss.item() * len(features)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_targets.append(targets.numpy())

    probs = np.concatenate(all_probs)
    targets = np.concatenate(all_targets)
    metrics = multilabel_subset_metrics(
        targets, probs, np.full(targets.shape[1], 0.5)
    )
    return {
        "loss": total_loss / len(loader.dataset),
        "macro_auc": metrics["macro_auroc"],
        "micro_auc": metrics["micro_auroc"],
        "macro_f1": metrics["macro_f1_at_05"],
        "micro_f1": metrics["micro_f1_at_05"],
    }


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

residual_model = ResidualNeighborModel(
    base_model=model,
    embedding_dim=320,
    hidden_dim=256,
    output_dim=len(LABEL_COLS),
).to(device)
residual_criterion = nn.BCEWithLogitsLoss()
residual_optimizer = torch.optim.AdamW(
    residual_model.correction_net.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in residual_model.parameters()
    if parameter.requires_grad
)
print(residual_model)
print(f"Trainable correction parameters: {trainable_parameters:,}")

In [ ]:
RESIDUAL_NUM_EPOCHS = 30
RESIDUAL_PATIENCE = 5

# Epoch 0 is the exact baseline and remains a valid fallback checkpoint.
initial_residual_val = evaluate_residual(
    residual_model,
    residual_val_connected_loader,
    residual_criterion,
    device,
)
residual_best_val_auc = initial_residual_val["macro_auc"]
residual_best_state = {
    key: value.detach().cpu().clone()
    for key, value in residual_model.state_dict().items()
}
residual_patience_counter = 0
residual_history = [{"epoch": 0, **initial_residual_val}]
print(
    f"Epoch 00 | exact baseline | "
    f"connected val macro AUC={residual_best_val_auc:.4f}"
)

for epoch in range(1, RESIDUAL_NUM_EPOCHS + 1):
    train_loss = train_residual_one_epoch(
        residual_model,
        residual_train_loader,
        residual_criterion,
        residual_optimizer,
        device,
    )
    val_metrics = evaluate_residual(
        residual_model,
        residual_val_connected_loader,
        residual_criterion,
        device,
    )
    residual_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        **val_metrics,
    })
    print(
        f"Epoch {epoch:02d} | train loss={train_loss:.4f} | "
        f"connected val loss={val_metrics['loss']:.4f} | "
        f"macro AUC={val_metrics['macro_auc']:.4f} | "
        f"micro AUC={val_metrics['micro_auc']:.4f}"
    )

    if val_metrics["macro_auc"] > residual_best_val_auc:
        residual_best_val_auc = val_metrics["macro_auc"]
        residual_best_state = {
            key: value.detach().cpu().clone()
            for key, value in residual_model.state_dict().items()
        }
        residual_patience_counter = 0
    else:
        residual_patience_counter += 1
        if residual_patience_counter >= RESIDUAL_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

residual_model.load_state_dict(residual_best_state)
residual_model = residual_model.to(device)
print(f"Best connected-validation macro AUC: {residual_best_val_auc:.4f}")

In [ ]:
residual_val_probs, residual_val_targets, residual_val_has_neighbor = (
    get_residual_predictions(
        residual_model, residual_val_connected_loader, device
    )
)
residual_test_probs, residual_test_targets, residual_test_has_neighbor = (
    get_residual_predictions(residual_model, residual_test_loader, device)
)
if not np.array_equal(residual_test_targets, test_targets):
    raise ValueError("Residual and baseline test targets are not aligned.")
if not np.array_equal(residual_test_has_neighbor, test_has_train_neighbor):
    raise ValueError("Residual test neighbor mask is not aligned.")

no_neighbor_mask = ~residual_test_has_neighbor
no_neighbor_max_abs_diff = np.max(np.abs(
    residual_test_probs[no_neighbor_mask]
    - test_probs[no_neighbor_mask]
))
np.testing.assert_allclose(
    residual_test_probs[no_neighbor_mask],
    test_probs[no_neighbor_mask],
    rtol=0,
    atol=1e-7,
)
print(
    "Maximum probability difference on no-neighbor test proteins:",
    no_neighbor_max_abs_diff,
)

# Thresholds for connected proteins are tuned only on connected validation
# proteins. No-neighbor proteins retain the original baseline thresholds.
residual_connected_thresholds = find_best_thresholds(
    residual_val_probs, residual_val_targets
)
residual_conditional_pred = (
    residual_test_probs >= best_thresholds[None, :]
).astype(int)
residual_conditional_pred[residual_test_has_neighbor] = (
    residual_test_probs[residual_test_has_neighbor]
    >= residual_connected_thresholds[None, :]
).astype(int)

residual_overall_metrics = multilabel_subset_metrics(
    residual_test_targets, residual_test_probs, best_thresholds
)
residual_overall_metrics["macro_f1_conditional"] = f1_score(
    residual_test_targets,
    residual_conditional_pred,
    average="macro",
    zero_division=0,
)
residual_overall_metrics["micro_f1_conditional"] = f1_score(
    residual_test_targets,
    residual_conditional_pred,
    average="micro",
    zero_division=0,
)
display(pd.DataFrame(
    [residual_overall_metrics],
    index=["ESM2 + gated residual neighbor"],
).round(4))

In [ ]:
residual_subgroup_rows = []
for subset_name, subset_mask in test_subsets.items():
    thresholds = (
        residual_connected_thresholds
        if subset_name == "Has train neighbor"
        else best_thresholds
    )
    residual_subgroup_rows.append({
        "test_subset": subset_name,
        "model": "ESM2 + gated residual neighbor",
        **multilabel_subset_metrics(
            residual_test_targets[subset_mask],
            residual_test_probs[subset_mask],
            thresholds,
        ),
    })

all_subgroup_results = (
    pd.concat([
        subgroup_results.reset_index(),
        pd.DataFrame(residual_subgroup_rows),
    ], ignore_index=True)
    .set_index(["test_subset", "model"])
)
display(all_subgroup_results.round(4))

residual_vs_baseline = []
for subset_name in test_subsets:
    baseline_row = all_subgroup_results.loc[(subset_name, "ESM2 + MLP")]
    residual_row = all_subgroup_results.loc[
        (subset_name, "ESM2 + gated residual neighbor")
    ]
    residual_vs_baseline.append({
        "test_subset": subset_name,
        "n": int(baseline_row["n"]),
        **{
            f"delta_{metric}": residual_row[metric] - baseline_row[metric]
            for metric in metric_cols
        },
    })

residual_delta_df = pd.DataFrame(residual_vs_baseline).set_index("test_subset")
print("Delta = gated residual - ESM2 baseline.")
display(residual_delta_df.round(4))

### Interpretation of the gated-residual run

The strict fallback guarantee works: across all **4,522 no-neighbor test proteins**, the maximum absolute probability difference from the original ESM2 baseline is exactly **0.0**.

However, connected-validation Macro AUROC does not improve after training the correction network. Early stopping therefore selects the zero-correction epoch-0 checkpoint. Consequently, the selected residual model has the same ranking metrics as the ESM2 baseline on both test subsets. Changes in the reported connected-subset tuned F1 arise only from fitting a separate threshold on the connected validation subset, not from changed logits.

This model succeeds as a safe architecture—it cannot damage remote no-neighbor proteins—but the current fixed weighted-mean neighbor representation does not provide a validation-supported residual correction. The next graph model should learn which neighbors/messages to retain rather than receiving only one pre-averaged neighbor vector.

## Minimal learnable neighbor attention

For each query protein, retain at most $K=8$ training neighbors. The old score $fident\times mincov$ is used **only to choose the top-K candidates**, not as the final importance. For each retained edge, the model learns

$$s_{ij}=MLP_{attn}([z_i\Vert z_j\Vert fident_{ij}\Vert mincov_{ij}]), \qquad \alpha_{ij}=softmax_j(s_{ij}),$$

and aggregates $n_i=\sum_j\alpha_{ij}W_vz_j$. The result enters the same safe residual framework, $l_i=l_i^{base}+m_i g([z_i\Vert n_i])$. The base checkpoint is frozen, the correction output is zero-initialized, and a hard availability gate guarantees exact baseline predictions when no training neighbor exists. No validation-to-validation or test-to-test messages are allowed.

In [ ]:
ATTENTION_TOP_K = 8

attention_edges = candidate.loc[
    candidate["target"].astype(str).isin(train_accessions),
    ["query", "target", "fident", "min_cov"],
].copy()
attention_edges[["query", "target"]] = (
    attention_edges[["query", "target"]].astype(str)
)
attention_edges = attention_edges[
    attention_edges["query"] != attention_edges["target"]
].copy()
attention_edges["selection_score"] = (
    attention_edges["fident"] * attention_edges["min_cov"]
)
attention_edges = (
    attention_edges
    .sort_values(
        ["query", "selection_score", "fident", "min_cov", "target"],
        ascending=[True, False, False, False, True],
        kind="mergesort",
    )
    .drop_duplicates(["query", "target"], keep="first")
    .groupby("query", sort=False, group_keys=False)
    .head(ATTENTION_TOP_K)
    .copy()
)
attention_edges["neighbor_rank"] = (
    attention_edges.groupby("query", sort=False).cumcount()
)

attention_query_idx = attention_edges["query"].map(protein_index)
attention_target_idx = attention_edges["target"].map(protein_index)
if attention_query_idx.isna().any() or attention_target_idx.isna().any():
    raise ValueError("A top-K edge contains an accession absent from the dataset.")

topk_neighbor_indices = np.zeros(
    (len(df), ATTENTION_TOP_K), dtype=np.int64
)
topk_edge_features = np.zeros(
    (len(df), ATTENTION_TOP_K, 2), dtype=np.float32
)
topk_neighbor_mask = np.zeros(
    (len(df), ATTENTION_TOP_K), dtype=bool
)

query_positions = attention_query_idx.to_numpy(int)
neighbor_positions = attention_edges["neighbor_rank"].to_numpy(int)
topk_neighbor_indices[query_positions, neighbor_positions] = (
    attention_target_idx.to_numpy(int)
)
topk_edge_features[query_positions, neighbor_positions, 0] = (
    attention_edges["fident"].to_numpy(np.float32)
)
topk_edge_features[query_positions, neighbor_positions, 1] = (
    attention_edges["min_cov"].to_numpy(np.float32)
)
topk_neighbor_mask[query_positions, neighbor_positions] = True

topk_has_neighbor = topk_neighbor_mask.any(axis=1)
if not np.array_equal(topk_has_neighbor, has_train_neighbor):
    raise ValueError("Top-K and fixed-aggregation neighbor availability disagree.")

topk_diagnostics = (
    pd.DataFrame({
        "split": df["split_homology"],
        "has_train_neighbor": topk_has_neighbor,
        "retained_neighbors": topk_neighbor_mask.sum(axis=1),
    })
    .groupby("split", sort=False)
    .agg(
        proteins=("has_train_neighbor", "size"),
        connected=("has_train_neighbor", "sum"),
        mean_retained_neighbors=("retained_neighbors", "mean"),
    )
)
print(f"Top-K directed edges retained: {len(attention_edges):,}")
display(topk_diagnostics)

In [ ]:
import copy

class LearnableNeighborAttentionResidual(nn.Module):
    def __init__(
        self,
        base_model,
        embedding_dim=320,
        attention_hidden_dim=128,
        correction_hidden_dim=256,
        output_dim=11,
    ):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.base_model = copy.deepcopy(base_model)
        for parameter in self.base_model.parameters():
            parameter.requires_grad = False

        self.attention_mlp = nn.Sequential(
            nn.Linear(2 * embedding_dim + 2, attention_hidden_dim),
            nn.ReLU(),
            nn.Linear(attention_hidden_dim, 1),
        )
        self.value_projection = nn.Linear(
            embedding_dim, embedding_dim, bias=False
        )
        nn.init.eye_(self.value_projection.weight)

        self.correction_net = nn.Sequential(
            nn.Linear(2 * embedding_dim, correction_hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(correction_hidden_dim, output_dim),
        )
        nn.init.zeros_(self.correction_net[-1].weight)
        nn.init.zeros_(self.correction_net[-1].bias)

    def train(self, mode=True):
        super().train(mode)
        self.base_model.eval()
        return self

    def forward(
        self,
        query_embeddings,
        neighbor_embeddings,
        edge_features,
        neighbor_mask,
        return_attention=False,
    ):
        batch_size, n_neighbors, _ = neighbor_embeddings.shape
        expanded_query = query_embeddings[:, None, :].expand(
            -1, n_neighbors, -1
        )
        attention_inputs = torch.cat(
            [expanded_query, neighbor_embeddings, edge_features], dim=-1
        )
        attention_scores = self.attention_mlp(attention_inputs).squeeze(-1)
        masked_scores = attention_scores.masked_fill(~neighbor_mask, -1e9)
        attention_weights = torch.softmax(masked_scores, dim=1)
        attention_weights = attention_weights * neighbor_mask.to(
            attention_weights.dtype
        )
        attention_weights = attention_weights / attention_weights.sum(
            dim=1, keepdim=True
        ).clamp_min(1e-12)

        neighbor_values = self.value_projection(neighbor_embeddings)
        aggregated_neighbor = torch.sum(
            attention_weights[:, :, None] * neighbor_values, dim=1
        )
        with torch.no_grad():
            base_logits = self.base_model(query_embeddings)
        delta_logits = self.correction_net(
            torch.cat([query_embeddings, aggregated_neighbor], dim=1)
        )
        availability_gate = neighbor_mask.any(dim=1).to(
            delta_logits.dtype
        )[:, None]
        logits = base_logits + availability_gate * delta_logits

        if return_attention:
            return logits, attention_weights
        return logits


def make_attention_loader(row_mask, batch_size, shuffle):
    row_indices = np.flatnonzero(np.asarray(row_mask, dtype=bool))
    return DataLoader(
        TensorDataset(
            torch.from_numpy(row_indices.astype(np.int64)),
            torch.from_numpy(y[row_indices]),
            torch.from_numpy(topk_neighbor_indices[row_indices]),
            torch.from_numpy(topk_edge_features[row_indices]),
            torch.from_numpy(topk_neighbor_mask[row_indices]),
        ),
        batch_size=batch_size,
        shuffle=shuffle,
    )


attention_train_loader = make_attention_loader(
    connected_train_mask, batch_size=32, shuffle=True
)
attention_val_connected_loader = make_attention_loader(
    connected_val_mask, batch_size=32, shuffle=False
)
attention_test_loader = make_attention_loader(
    test_mask_np, batch_size=32, shuffle=False
)
embedding_bank = torch.from_numpy(X_esm)

In [ ]:
def attention_forward_batch(model, batch, embedding_bank, device, return_attention=False):
    query_idx, targets, neighbor_idx, edge_features, neighbor_mask = batch
    query_idx = query_idx.to(device)
    neighbor_idx = neighbor_idx.to(device)
    edge_features = edge_features.to(device)
    neighbor_mask = neighbor_mask.to(device)
    query_embeddings = embedding_bank[query_idx]
    neighbor_embeddings = embedding_bank[neighbor_idx]
    output = model(
        query_embeddings,
        neighbor_embeddings,
        edge_features,
        neighbor_mask,
        return_attention=return_attention,
    )
    return output, targets, query_idx, neighbor_mask


def train_attention_one_epoch(
    model, loader, embedding_bank, criterion, optimizer, device
):
    model.train()
    total_loss = 0.0
    for batch in loader:
        logits, targets, _, _ = attention_forward_batch(
            model, batch, embedding_bank, device
        )
        targets = targets.to(device)
        optimizer.zero_grad()
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(targets)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def get_attention_predictions(model, loader, embedding_bank, device):
    model.eval()
    all_probs, all_targets, all_query_idx = [], [], []
    all_neighbor_masks, all_attention = [], []

    for batch in loader:
        output, targets, query_idx, neighbor_mask = attention_forward_batch(
            model, batch, embedding_bank, device, return_attention=True
        )
        logits, attention_weights = output
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_targets.append(targets.numpy())
        all_query_idx.append(query_idx.cpu().numpy())
        all_neighbor_masks.append(neighbor_mask.cpu().numpy())
        all_attention.append(attention_weights.cpu().numpy())

    return (
        np.concatenate(all_probs),
        np.concatenate(all_targets),
        np.concatenate(all_query_idx),
        np.concatenate(all_neighbor_masks),
        np.concatenate(all_attention),
    )


@torch.no_grad()
def evaluate_attention(model, loader, embedding_bank, criterion, device):
    model.eval()
    total_loss = 0.0
    all_probs, all_targets = [], []
    for batch in loader:
        logits, targets, _, _ = attention_forward_batch(
            model, batch, embedding_bank, device
        )
        targets_device = targets.to(device)
        loss = criterion(logits, targets_device)
        total_loss += loss.item() * len(targets)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_targets.append(targets.numpy())

    probs = np.concatenate(all_probs)
    targets = np.concatenate(all_targets)
    metrics = multilabel_subset_metrics(
        targets, probs, np.full(targets.shape[1], 0.5)
    )
    return {
        "loss": total_loss / len(loader.dataset),
        "macro_auc": metrics["macro_auroc"],
        "micro_auc": metrics["micro_auroc"],
        "macro_f1": metrics["macro_f1_at_05"],
        "micro_f1": metrics["micro_f1_at_05"],
    }


# Previous experiments no longer need GPU-resident model/optimizer state.
# Releasing it keeps this section runnable top-to-bottom on modest GPUs.
import gc
model = model.cpu()
if "fixed_model" in globals():
    fixed_model = fixed_model.cpu()
if "residual_model" in globals():
    residual_model = residual_model.cpu()
for optimizer_name in ["optimizer", "fixed_optimizer", "residual_optimizer"]:
    globals().pop(optimizer_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
attention_model = LearnableNeighborAttentionResidual(
    base_model=model,
    embedding_dim=320,
    attention_hidden_dim=128,
    correction_hidden_dim=256,
    output_dim=len(LABEL_COLS),
).to(device)
embedding_bank = embedding_bank.to(device)
attention_criterion = nn.BCEWithLogitsLoss()
attention_optimizer = torch.optim.AdamW(
    [
        parameter
        for parameter in attention_model.parameters()
        if parameter.requires_grad
    ],
    lr=1e-3,
    weight_decay=1e-4,
)
attention_trainable_parameters = sum(
    parameter.numel()
    for parameter in attention_model.parameters()
    if parameter.requires_grad
)
print(attention_model)
print(f"Trainable attention/correction parameters: {attention_trainable_parameters:,}")

In [ ]:
ATTENTION_NUM_EPOCHS = 30
ATTENTION_PATIENCE = 5

initial_attention_val = evaluate_attention(
    attention_model,
    attention_val_connected_loader,
    embedding_bank,
    attention_criterion,
    device,
)
attention_best_val_auc = initial_attention_val["macro_auc"]
attention_best_state = {
    key: value.detach().cpu().clone()
    for key, value in attention_model.state_dict().items()
}
attention_patience_counter = 0
attention_history = [{"epoch": 0, **initial_attention_val}]
print(
    f"Epoch 00 | exact baseline | "
    f"connected val macro AUC={attention_best_val_auc:.4f}"
)

for epoch in range(1, ATTENTION_NUM_EPOCHS + 1):
    train_loss = train_attention_one_epoch(
        attention_model,
        attention_train_loader,
        embedding_bank,
        attention_criterion,
        attention_optimizer,
        device,
    )
    val_metrics = evaluate_attention(
        attention_model,
        attention_val_connected_loader,
        embedding_bank,
        attention_criterion,
        device,
    )
    attention_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        **val_metrics,
    })
    print(
        f"Epoch {epoch:02d} | train loss={train_loss:.4f} | "
        f"connected val loss={val_metrics['loss']:.4f} | "
        f"macro AUC={val_metrics['macro_auc']:.4f} | "
        f"micro AUC={val_metrics['micro_auc']:.4f}"
    )

    if val_metrics["macro_auc"] > attention_best_val_auc:
        attention_best_val_auc = val_metrics["macro_auc"]
        attention_best_state = {
            key: value.detach().cpu().clone()
            for key, value in attention_model.state_dict().items()
        }
        attention_patience_counter = 0
    else:
        attention_patience_counter += 1
        if attention_patience_counter >= ATTENTION_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

attention_model.load_state_dict(attention_best_state)
attention_model = attention_model.to(device)
print(f"Best connected-validation macro AUC: {attention_best_val_auc:.4f}")

In [ ]:
(
    attention_val_probs,
    attention_val_targets,
    attention_val_query_idx,
    attention_val_neighbor_mask,
    attention_val_weights,
) = get_attention_predictions(
    attention_model, attention_val_connected_loader, embedding_bank, device
)
(
    attention_test_probs,
    attention_test_targets,
    attention_test_query_idx,
    attention_test_neighbor_mask,
    attention_test_weights,
) = get_attention_predictions(
    attention_model, attention_test_loader, embedding_bank, device
)

expected_test_idx = np.flatnonzero(test_mask_np)
if not np.array_equal(attention_test_query_idx, expected_test_idx):
    raise ValueError("Attention predictions are not in dataset test order.")
if not np.array_equal(attention_test_targets, test_targets):
    raise ValueError("Attention and baseline test targets are not aligned.")
attention_test_has_neighbor = attention_test_neighbor_mask.any(axis=1)
if not np.array_equal(attention_test_has_neighbor, test_has_train_neighbor):
    raise ValueError("Attention test neighbor mask is not aligned.")

@torch.no_grad()
def get_matching_batch_base_probs(model, loader, embedding_bank, device):
    model.eval()
    all_base_probs = []
    for query_idx, _, _, _, _ in loader:
        query_embeddings = embedding_bank[query_idx.to(device)]
        base_logits = model.base_model(query_embeddings)
        all_base_probs.append(torch.sigmoid(base_logits).cpu().numpy())
    return np.concatenate(all_base_probs)

attention_base_test_probs = get_matching_batch_base_probs(
    attention_model, attention_test_loader, embedding_bank, device
)
attention_no_neighbor_mask = ~attention_test_has_neighbor
attention_internal_gate_diff = np.max(np.abs(
    attention_test_probs[attention_no_neighbor_mask]
    - attention_base_test_probs[attention_no_neighbor_mask]
))
np.testing.assert_array_equal(
    attention_test_probs[attention_no_neighbor_mask],
    attention_base_test_probs[attention_no_neighbor_mask],
)
attention_original_baseline_diff = np.max(np.abs(
    attention_test_probs[attention_no_neighbor_mask]
    - test_probs[attention_no_neighbor_mask]
))
np.testing.assert_allclose(
    attention_test_probs[attention_no_neighbor_mask],
    test_probs[attention_no_neighbor_mask],
    rtol=0,
    atol=1e-6,
)
print("Exact within-batch gate difference:", attention_internal_gate_diff)
print(
    "Difference versus earlier baseline batches:",
    attention_original_baseline_diff,
)

attention_connected_thresholds = find_best_thresholds(
    attention_val_probs, attention_val_targets
)
attention_conditional_pred = (
    attention_test_probs >= best_thresholds[None, :]
).astype(int)
attention_conditional_pred[attention_test_has_neighbor] = (
    attention_test_probs[attention_test_has_neighbor]
    >= attention_connected_thresholds[None, :]
).astype(int)

attention_overall_metrics = multilabel_subset_metrics(
    attention_test_targets, attention_test_probs, best_thresholds
)
attention_overall_metrics["macro_f1_conditional"] = f1_score(
    attention_test_targets,
    attention_conditional_pred,
    average="macro",
    zero_division=0,
)
attention_overall_metrics["micro_f1_conditional"] = f1_score(
    attention_test_targets,
    attention_conditional_pred,
    average="micro",
    zero_division=0,
)
display(pd.DataFrame(
    [attention_overall_metrics],
    index=["ESM2 + learnable neighbor attention"],
).round(4))

In [ ]:
attention_subgroup_rows = []
for subset_name, subset_mask in test_subsets.items():
    thresholds = (
        attention_connected_thresholds
        if subset_name == "Has train neighbor"
        else best_thresholds
    )
    attention_subgroup_rows.append({
        "test_subset": subset_name,
        "model": "ESM2 + learnable neighbor attention",
        **multilabel_subset_metrics(
            attention_test_targets[subset_mask],
            attention_test_probs[subset_mask],
            thresholds,
        ),
    })

all_attention_subgroup_results = (
    pd.concat([
        all_subgroup_results.reset_index(),
        pd.DataFrame(attention_subgroup_rows),
    ], ignore_index=True)
    .set_index(["test_subset", "model"])
)
display(all_attention_subgroup_results.round(4))

attention_vs_baseline = []
for subset_name in test_subsets:
    baseline_row = all_attention_subgroup_results.loc[
        (subset_name, "ESM2 + MLP")
    ]
    attention_row = all_attention_subgroup_results.loc[
        (subset_name, "ESM2 + learnable neighbor attention")
    ]
    attention_vs_baseline.append({
        "test_subset": subset_name,
        "n": int(baseline_row["n"]),
        **{
            f"delta_{metric}": attention_row[metric] - baseline_row[metric]
            for metric in metric_cols
        },
    })

attention_delta_df = pd.DataFrame(attention_vs_baseline).set_index(
    "test_subset"
)
print("Delta = learnable attention - ESM2 baseline.")
display(attention_delta_df.round(4))

In [ ]:
connected_attention_sums = attention_test_weights[
    attention_test_has_neighbor
].sum(axis=1)
np.testing.assert_allclose(
    connected_attention_sums,
    np.ones_like(connected_attention_sums),
    rtol=1e-5,
    atol=1e-6,
)
assert np.all(
    attention_test_weights[~attention_test_neighbor_mask] == 0
)
print("Connected attention sums:", connected_attention_sums.min(), connected_attention_sums.max())

# Show one connected test query with the maximum retained neighbor count.
test_neighbor_counts = attention_test_neighbor_mask.sum(axis=1)
example_test_position = int(np.argmax(test_neighbor_counts))
example_query_idx = attention_test_query_idx[example_test_position]
example_valid = attention_test_neighbor_mask[example_test_position]
example_neighbor_idx = topk_neighbor_indices[example_query_idx, example_valid]
example_edges = topk_edge_features[example_query_idx, example_valid]
example_attention = attention_test_weights[example_test_position, example_valid]
attention_example_df = pd.DataFrame({
    "query": dataset_acc[example_query_idx],
    "neighbor": dataset_acc[example_neighbor_idx],
    "fident": example_edges[:, 0],
    "min_cov": example_edges[:, 1],
    "selection_score": example_edges[:, 0] * example_edges[:, 1],
    "learned_attention": example_attention,
}).sort_values("learned_attention", ascending=False)
display(attention_example_df.round(4))

### Interpretation of the learnable-attention run

The connected-validation selector chooses **epoch 1** rather than the zero-correction fallback, but the improvement is extremely small. On the **940 connected test proteins**, Macro AUROC changes from **0.900494 to 0.900587** ($+0.000093$), Macro AUPRC from **0.618158 to 0.618835** ($+0.000678$), Macro F1 at 0.5 from **0.500057 to 0.515848** ($+0.015791$), and Micro F1 at 0.5 from **0.682240 to 0.689580** ($+0.007340$). Validation-tuned Macro F1 decreases by $0.018810$, while tuned Micro F1 increases by $0.007496$.

For all **4,522 no-neighbor test proteins**, every metric is unchanged. The within-batch gated output is bitwise identical to the frozen baseline (maximum difference **0.0**); comparison with the earlier batch-512 baseline differs by at most $1.79\times10^{-7}$ because CUDA matrix multiplication uses a different batch shape.

The attention weights are valid and sum to one over retained neighbors, but the inspected example remains close to uniform. Overall, learnable attention produces a directionally positive but very small ranking signal on connected proteins. This is not yet strong evidence of a practically meaningful improvement and should be checked across multiple seeds.

## Does the graph help a weak sequence representation? AAC control

Each protein is represented only by its 20-dimensional amino-acid composition (AAC), $x_i=[p_A,p_C,\ldots,p_Y]$. We compare the same four conditions under the same homology split:

| Representation | No graph | + Graph |
|---|---|---|
| AAC | AAC + MLP | AAC + learnable homology attention |
| ESM2 | ESM2 + MLP | ESM2 + learnable homology attention |

The AAC graph model reuses the exact same top-8 training-neighbor tensors, the same two edge features, and the same frozen-base residual safety mechanism. This isolates whether explicit homology is more valuable when the node representation itself is weak.

In [ ]:
from collections import Counter

AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
aa_to_index = {amino_acid: idx for idx, amino_acid in enumerate(AMINO_ACIDS)}
aac_features = np.zeros((len(df), len(AMINO_ACIDS)), dtype=np.float32)
nonstandard_residue_count = 0
total_residue_count = 0

for protein_idx, sequence in enumerate(df["Sequence"].astype(str).str.upper()):
    if len(sequence) == 0:
        raise ValueError(f"Empty sequence at dataset row {protein_idx}.")
    counts = Counter(sequence)
    sequence_length = len(sequence)
    total_residue_count += sequence_length
    standard_count = 0
    for amino_acid, feature_idx in aa_to_index.items():
        count = counts.get(amino_acid, 0)
        standard_count += count
        aac_features[protein_idx, feature_idx] = count / sequence_length
    nonstandard_residue_count += sequence_length - standard_count

if not np.isfinite(aac_features).all():
    raise ValueError("AAC features contain non-finite values.")
if np.any(aac_features < 0) or np.any(aac_features.sum(axis=1) > 1.000001):
    raise ValueError("AAC feature proportions are invalid.")

print("AAC feature matrix:", aac_features.shape, aac_features.dtype)
print(
    "Non-standard residue fraction:",
    nonstandard_residue_count / total_residue_count,
)
display(pd.DataFrame(
    {"mean_frequency": aac_features.mean(axis=0)},
    index=AMINO_ACIDS,
).T.round(4))

In [ ]:
import gc

# The ESM2-attention experiment is complete; release its GPU state before
# training the independent AAC control.
if "attention_model" in globals():
    attention_model = attention_model.cpu()
if "embedding_bank" in globals():
    embedding_bank = embedding_bank.cpu()
globals().pop("attention_optimizer", None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

aac_train_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(aac_features[train_mask_np]),
        torch.from_numpy(y[train_mask_np]),
    ),
    batch_size=256,
    shuffle=True,
)
aac_val_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(aac_features[val_mask_np]),
        torch.from_numpy(y[val_mask_np]),
    ),
    batch_size=512,
    shuffle=False,
)
aac_test_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(aac_features[test_mask_np]),
        torch.from_numpy(y[test_mask_np]),
    ),
    batch_size=512,
    shuffle=False,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
aac_model = ProteinMLP(
    input_dim=len(AMINO_ACIDS),
    hidden_dim=256,
    output_dim=len(LABEL_COLS),
).to(device)
aac_criterion = nn.BCEWithLogitsLoss()
aac_optimizer = torch.optim.AdamW(
    aac_model.parameters(), lr=1e-3, weight_decay=1e-4
)

AAC_NUM_EPOCHS = 30
AAC_PATIENCE = 5
aac_best_val_auc = -np.inf
aac_best_state = None
aac_patience_counter = 0
aac_history = []

for epoch in range(1, AAC_NUM_EPOCHS + 1):
    train_loss = train_one_epoch(
        aac_model, aac_train_loader, aac_criterion, aac_optimizer, device
    )
    val_metrics = evaluate(
        aac_model, aac_val_loader, aac_criterion, device
    )
    aac_history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})
    print(
        f"Epoch {epoch:02d} | train loss={train_loss:.4f} | "
        f"val loss={val_metrics['loss']:.4f} | "
        f"macro AUC={val_metrics['macro_auc']:.4f} | "
        f"micro AUC={val_metrics['micro_auc']:.4f}"
    )
    if val_metrics["macro_auc"] > aac_best_val_auc:
        aac_best_val_auc = val_metrics["macro_auc"]
        aac_best_state = {
            key: value.detach().cpu().clone()
            for key, value in aac_model.state_dict().items()
        }
        aac_patience_counter = 0
    else:
        aac_patience_counter += 1
        if aac_patience_counter >= AAC_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

if aac_best_state is None:
    raise RuntimeError("AAC baseline training produced no valid checkpoint.")
aac_model.load_state_dict(aac_best_state)
aac_model = aac_model.to(device)
print(f"Best AAC validation macro AUC: {aac_best_val_auc:.4f}")

In [ ]:
aac_test_metrics = evaluate(
    aac_model, aac_test_loader, aac_criterion, device
)
aac_val_probs, aac_val_targets = get_predictions(
    aac_model, aac_val_loader, device
)
aac_test_probs, aac_test_targets = get_predictions(
    aac_model, aac_test_loader, device
)
if not np.array_equal(aac_test_targets, test_targets):
    raise ValueError("AAC and ESM2 test targets are not aligned.")
aac_best_thresholds = find_best_thresholds(aac_val_probs, aac_val_targets)
aac_baseline_metrics = multilabel_subset_metrics(
    aac_test_targets, aac_test_probs, aac_best_thresholds
)
display(pd.DataFrame(
    [aac_baseline_metrics], index=["AAC + MLP"]
).round(4))

In [ ]:
aac_model = aac_model.cpu()
aac_attention_device = torch.device("cpu")
globals().pop("aac_optimizer", None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
aac_attention_model = LearnableNeighborAttentionResidual(
    base_model=aac_model,
    embedding_dim=len(AMINO_ACIDS),
    attention_hidden_dim=128,
    correction_hidden_dim=256,
    output_dim=len(LABEL_COLS),
).to(aac_attention_device)
aac_embedding_bank = torch.from_numpy(aac_features).to(aac_attention_device)
aac_attention_criterion = nn.BCEWithLogitsLoss()
aac_attention_optimizer = torch.optim.AdamW(
    [
        parameter
        for parameter in aac_attention_model.parameters()
        if parameter.requires_grad
    ],
    lr=1e-3,
    weight_decay=1e-4,
)

AAC_ATTENTION_NUM_EPOCHS = 30
AAC_ATTENTION_PATIENCE = 5
initial_aac_attention_val = evaluate_attention(
    aac_attention_model,
    attention_val_connected_loader,
    aac_embedding_bank,
    aac_attention_criterion,
    aac_attention_device,
)
aac_attention_best_val_auc = initial_aac_attention_val["macro_auc"]
aac_attention_best_state = {
    key: value.detach().cpu().clone()
    for key, value in aac_attention_model.state_dict().items()
}
aac_attention_patience_counter = 0
aac_attention_history = [{"epoch": 0, **initial_aac_attention_val}]
print(
    f"Epoch 00 | exact AAC baseline | "
    f"connected val macro AUC={aac_attention_best_val_auc:.4f}"
)

for epoch in range(1, AAC_ATTENTION_NUM_EPOCHS + 1):
    train_loss = train_attention_one_epoch(
        aac_attention_model,
        attention_train_loader,
        aac_embedding_bank,
        aac_attention_criterion,
        aac_attention_optimizer,
        aac_attention_device,
    )
    val_metrics = evaluate_attention(
        aac_attention_model,
        attention_val_connected_loader,
        aac_embedding_bank,
        aac_attention_criterion,
        aac_attention_device,
    )
    aac_attention_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        **val_metrics,
    })
    print(
        f"Epoch {epoch:02d} | train loss={train_loss:.4f} | "
        f"connected val loss={val_metrics['loss']:.4f} | "
        f"macro AUC={val_metrics['macro_auc']:.4f} | "
        f"micro AUC={val_metrics['micro_auc']:.4f}"
    )
    if val_metrics["macro_auc"] > aac_attention_best_val_auc:
        aac_attention_best_val_auc = val_metrics["macro_auc"]
        aac_attention_best_state = {
            key: value.detach().cpu().clone()
            for key, value in aac_attention_model.state_dict().items()
        }
        aac_attention_patience_counter = 0
    else:
        aac_attention_patience_counter += 1
        if aac_attention_patience_counter >= AAC_ATTENTION_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

aac_attention_model.load_state_dict(aac_attention_best_state)
aac_attention_model = aac_attention_model.to(aac_attention_device)
print(
    f"Best AAC-attention connected-validation macro AUC: "
    f"{aac_attention_best_val_auc:.4f}"
)

In [ ]:
(
    aac_attention_val_probs,
    aac_attention_val_targets,
    _,
    _,
    _,
) = get_attention_predictions(
    aac_attention_model,
    attention_val_connected_loader,
    aac_embedding_bank,
    aac_attention_device,
)
(
    aac_attention_test_probs,
    aac_attention_test_targets,
    aac_attention_test_query_idx,
    aac_attention_test_neighbor_mask,
    aac_attention_test_weights,
) = get_attention_predictions(
    aac_attention_model, attention_test_loader, aac_embedding_bank, aac_attention_device
)
if not np.array_equal(aac_attention_test_targets, aac_test_targets):
    raise ValueError("AAC attention and AAC baseline targets are not aligned.")

aac_attention_base_batches = []
with torch.no_grad():
    aac_attention_model.eval()
    for query_idx, _, _, _, _ in attention_test_loader:
        query_embeddings = aac_embedding_bank[query_idx.to(aac_attention_device)]
        base_logits = aac_attention_model.base_model(query_embeddings)
        aac_attention_base_batches.append(
            torch.sigmoid(base_logits).cpu().numpy()
        )
aac_attention_base_test_probs = np.concatenate(aac_attention_base_batches)
aac_attention_test_has_neighbor = aac_attention_test_neighbor_mask.any(axis=1)
aac_no_neighbor_mask = ~aac_attention_test_has_neighbor
aac_internal_gate_diff = np.max(np.abs(
    aac_attention_test_probs[aac_no_neighbor_mask]
    - aac_attention_base_test_probs[aac_no_neighbor_mask]
))
np.testing.assert_array_equal(
    aac_attention_test_probs[aac_no_neighbor_mask],
    aac_attention_base_test_probs[aac_no_neighbor_mask],
)
print("AAC exact within-batch gate difference:", aac_internal_gate_diff)

aac_attention_connected_thresholds = find_best_thresholds(
    aac_attention_val_probs, aac_attention_val_targets
)
aac_attention_conditional_pred = (
    aac_attention_test_probs >= aac_best_thresholds[None, :]
).astype(int)
aac_attention_conditional_pred[aac_attention_test_has_neighbor] = (
    aac_attention_test_probs[aac_attention_test_has_neighbor]
    >= aac_attention_connected_thresholds[None, :]
).astype(int)
aac_attention_overall_metrics = multilabel_subset_metrics(
    aac_attention_test_targets,
    aac_attention_test_probs,
    aac_best_thresholds,
)
aac_attention_overall_metrics["macro_f1_conditional"] = f1_score(
    aac_attention_test_targets,
    aac_attention_conditional_pred,
    average="macro",
    zero_division=0,
)
aac_attention_overall_metrics["micro_f1_conditional"] = f1_score(
    aac_attention_test_targets,
    aac_attention_conditional_pred,
    average="micro",
    zero_division=0,
)
display(pd.DataFrame(
    [aac_attention_overall_metrics],
    index=["AAC + learnable homology attention"],
).round(4))

In [ ]:
two_by_two_macro_auroc = pd.DataFrame(
    {
        "No graph": [
            aac_baseline_metrics["macro_auroc"],
            test_metrics["macro_auc"],
        ],
        "+ Graph": [
            aac_attention_overall_metrics["macro_auroc"],
            attention_overall_metrics["macro_auroc"],
        ],
    },
    index=["AAC", "ESM2"],
)
two_by_two_macro_auroc["Graph delta"] = (
    two_by_two_macro_auroc["+ Graph"]
    - two_by_two_macro_auroc["No graph"]
)
print("Overall test Macro AUROC")
display(two_by_two_macro_auroc.round(4))

representation_graph_details = pd.DataFrame([
    {"representation": "AAC", "graph": False, **aac_baseline_metrics},
    {
        "representation": "AAC",
        "graph": True,
        **aac_attention_overall_metrics,
    },
    {
        "representation": "ESM2",
        "graph": False,
        "n": len(test_targets),
        "macro_auroc": test_metrics["macro_auc"],
        "micro_auroc": test_metrics["micro_auc"],
        "macro_auprc": macro_auprc,
        "micro_auprc": micro_auprc,
        "macro_f1_at_05": test_metrics["macro_f1"],
        "micro_f1_at_05": test_metrics["micro_f1"],
        "macro_f1_tuned": macro_f1_tuned,
        "micro_f1_tuned": micro_f1_tuned,
    },
    {
        "representation": "ESM2",
        "graph": True,
        **attention_overall_metrics,
    },
])
display(
    representation_graph_details
    .set_index(["representation", "graph"])
    .round(4)
)

In [ ]:
aac_subgroup_rows = []
for subset_name, subset_mask in test_subsets.items():
    baseline_subset = multilabel_subset_metrics(
        aac_test_targets[subset_mask],
        aac_test_probs[subset_mask],
        aac_best_thresholds,
    )
    graph_thresholds = (
        aac_attention_connected_thresholds
        if subset_name == "Has train neighbor"
        else aac_best_thresholds
    )
    graph_subset = multilabel_subset_metrics(
        aac_attention_test_targets[subset_mask],
        aac_attention_test_probs[subset_mask],
        graph_thresholds,
    )
    aac_subgroup_rows.extend([
        {"test_subset": subset_name, "model": "AAC + MLP", **baseline_subset},
        {
            "test_subset": subset_name,
            "model": "AAC + homology attention",
            **graph_subset,
        },
    ])

aac_subgroup_results = pd.DataFrame(aac_subgroup_rows).set_index(
    ["test_subset", "model"]
)
display(aac_subgroup_results.round(4))

aac_connected_base = aac_subgroup_results.loc[
    ("Has train neighbor", "AAC + MLP")
]
aac_connected_graph = aac_subgroup_results.loc[
    ("Has train neighbor", "AAC + homology attention")
]
aac_connected_delta = pd.Series(
    {
        metric: aac_connected_graph[metric] - aac_connected_base[metric]
        for metric in metric_cols
    },
    name="Graph - no graph on connected AAC test proteins",
)
display(aac_connected_delta.to_frame().T.round(4))

### Interpretation of the AAC control

The 2×2 control supports a representation-dependent role for explicit homology. On the full test set, AAC Macro AUROC increases from **0.7035 to 0.7109** ($+0.0074$), whereas ESM2 changes from **0.8946 to 0.8945** (approximately no improvement). The full-test AAC effect is diluted because only 940 of 5,462 test proteins can receive a training-neighbor message.

On those **940 connected test proteins**, the AAC effect is large and internally consistent:

- Macro AUROC: **0.7050 → 0.7816** ($+0.0766$)
- Micro AUROC: **0.8142 → 0.8672** ($+0.0530$)
- Macro AUPRC: **0.2715 → 0.3984** ($+0.1268$)
- Micro AUPRC: **0.4922 → 0.5940** ($+0.1018$)
- Validation-tuned Macro F1: **0.2988 → 0.4134** ($+0.1146$)
- Validation-tuned Micro F1: **0.4274 → 0.5173** ($+0.0899$)

All metrics for the 4,522 no-neighbor proteins are unchanged, confirming the residual safety property. The strongest defensible conclusion from this seed is: **explicit homology strongly complements weak global-composition features, but adds little beyond pretrained ESM2 representations under this split and graph coverage**. Multiple seeds or paired bootstrap intervals are still required for a statistical claim.